In [1]:
import datetime as dt
import sys
import time
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

from utilities.preprocessor import *
from utilities.values import *

data_dir = '../data/'
model_dir = '../models/'

### 1. Merge the separate raw datafiles

In [2]:
# combine all the raw files
raw_data_dir_paths = [data_dir + 'raw/2020', data_dir + 'raw/2021',]
full_dataset_df = read_all_files_in_dir(raw_data_dir_paths)

In [3]:
# force starttime into datetime format so that it is usable
full_dataset_df['starttime'] = pd.to_datetime(full_dataset_df['starttime'])

# the Feb-Apr 2021 files are in UTC, while all earlier files are in New York local time, so convert them
is_utc = full_dataset_df['starttime'] >= '2021-02-01'
full_dataset_df.loc[is_utc, 'starttime'] = (
    full_dataset_df.loc[is_utc, 'starttime']
    .dt.tz_localize('UTC')
    .dt.tz_convert(NYC_TIMEZONE)
    .dt.tz_localize(None)
)

# remove duplicate trips (corrected Feb 2021 file overlaps with Jan 2021 files)
full_dataset_df = full_dataset_df.drop_duplicates(subset = ['bikeid', 'starttime', 'start station name'])

# sort by increasing start and end dates
full_dataset_df = full_dataset_df.sort_values(['starttime', 'stoptime'])

print('Total number of rows:', len(full_dataset_df))
print('Data period start:', full_dataset_df['starttime'].min())
print('Data period end:' , full_dataset_df['starttime'].max())

Total number of rows: 379848
Data period start: 2020-01-01 00:04:50.192000
Data period end: 2021-04-30 19:59:43.247000


### 2. Preprocessing the dataset

#### 2a. Extract date features and get the rental counts for each station, date and time

In [4]:
extractor = DateFeatureExtractor()

# remove the depot, which only has 1 rental and is not an actual station
full_dataset_df = full_dataset_df[full_dataset_df[STATION] != 'JCBS Depot']

# extract hour first, so that we can make sure that each station-date-hour has a value
full_dataset_df[DATE] = full_dataset_df['starttime'].apply(extractor.to_date)
full_dataset_df[HOUR_OF_DAY] = full_dataset_df['starttime'].apply(extractor.to_hour_of_day)

In [5]:
# count rentals for each station-date-hour (only includes those with at least 1 rental)
rental_counts_df = full_dataset_df.groupby([STATION, DATE, HOUR_OF_DAY]).size().reset_index(name = NUM_RENTALS)

# make sure that every station-date-hour has a row, even if 0
stations = rental_counts_df[STATION].unique()
dates = pd.date_range('2020-01-01', '2021-04-30', freq = 'D').date
hours = range(24)

full_rental_counts_df = pd.MultiIndex.from_product(
    [stations, dates, hours], names = [STATION, DATE, HOUR_OF_DAY]
).to_frame(index = False)

# the last raw file ends at 19:59 on 30 Apr 2021 (New York time), so there is no data for 8pm-11pm that day
no_data = (full_rental_counts_df[DATE] == dt.date(2021, 4, 30)) & (full_rental_counts_df[HOUR_OF_DAY] >= 20)
full_rental_counts_df = full_rental_counts_df[~no_data]

In [6]:
# merge the rental counts, fill rows with no values to 0
full_rental_counts_df = full_rental_counts_df.merge(rental_counts_df, on = [STATION, DATE, HOUR_OF_DAY], how = 'left')
full_rental_counts_df[NUM_RENTALS] = full_rental_counts_df[NUM_RENTALS].fillna(0).astype(int)

# merge the coordinates of each station
station_locations_df = full_dataset_df.groupby(STATION)[[LATITUDE, LONGITUDE]].first().reset_index()
full_rental_counts_df = full_rental_counts_df.merge(station_locations_df, on = STATION, how = 'left')

In [7]:
# extract the other date features
full_rental_counts_df[WEEK_OF_YEAR] = full_rental_counts_df[DATE].apply(extractor.to_week_of_year)
full_rental_counts_df[SINE_WEEK_OF_YEAR] = extractor.to_sine_week_of_year(full_rental_counts_df[WEEK_OF_YEAR])
full_rental_counts_df[COSINE_WEEK_OF_YEAR] = extractor.to_cosine_week_of_year(full_rental_counts_df[WEEK_OF_YEAR])
full_rental_counts_df[DAY_OF_WEEK] = full_rental_counts_df[DATE].apply(extractor.to_day_of_week)
full_rental_counts_df[IS_PUBLIC_HOLIDAY] = full_rental_counts_df[DATE].apply(extractor.to_is_public_holiday)

# drop week of year, no longer needed
full_rental_counts_df = full_rental_counts_df.drop([WEEK_OF_YEAR], axis=1)

# sort for easy reading
full_rental_counts_df = full_rental_counts_df.sort_values([STATION, DATE, HOUR_OF_DAY])

In [8]:
full_rental_counts_df.head()

,start station name,date,hour_of_day,num_rentals,start station latitude,start station longitude,sine_week_of_year,cosine_week_of_year,day_of_week,is_public_holiday
0,5 Corners Library,2020-01-01,0,0,40.734961,-74.059503,0.0,1.0,2,1
1,5 Corners Library,2020-01-01,1,0,40.734961,-74.059503,0.0,1.0,2,1
2,5 Corners Library,2020-01-01,2,0,40.734961,-74.059503,0.0,1.0,2,1
3,5 Corners Library,2020-01-01,3,0,40.734961,-74.059503,0.0,1.0,2,1
4,5 Corners Library,2020-01-01,4,0,40.734961,-74.059503,0.0,1.0,2,1


#### 2b. Create new features using the actual rental counts of the previous hours

Since we are predicting the demand 1 hour ahead, we can use T-3 and T-2 hours. <br>
For example, if we are predicting the demand for 12pm at 11am, we can use the actual rental numbers for 9am and 10am 

In [9]:
# timestamp of each row, so that the lags work across midnight
rental_timestamp = (
    pd.to_datetime(full_rental_counts_df[DATE])
    + pd.to_timedelta(full_rental_counts_df[HOUR_OF_DAY], unit = 'h')
)

# lookup table: rentals for each station at each hour
rentals_lookup = (
    full_rental_counts_df
    .assign(timestamp = rental_timestamp)
    .groupby([STATION, 'timestamp'])[NUM_RENTALS]
    .sum()
)
data_start = rental_timestamp.min()

for lag in [2, 3]:
    lag_timestamp = rental_timestamp - pd.Timedelta(hours = lag)
    lag_keys = pd.MultiIndex.from_arrays([full_rental_counts_df[STATION], lag_timestamp])

    # same station, exactly `lag` hours earlier; no row for that hour means 0 rentals
    lag_values = pd.Series(
        rentals_lookup.reindex(lag_keys).fillna(0).to_numpy(),
        index = full_rental_counts_df.index,
    )

    # hours before the dataset begins have no data, so set them to 0
    full_rental_counts_df[f'{NUM_RENTALS}_t-{lag}'] = lag_values.where(lag_timestamp >= data_start, 0)


# save to file
full_rental_counts_df.to_csv(data_dir + 'intermediate_processing/full_rental_counts.csv', index=False)
print('Total number of rows:', len(full_rental_counts_df))

Total number of rows: 594660


In [10]:
full_rental_counts_df.head()

,start station name,date,hour_of_day,num_rentals,start station latitude,start station longitude,sine_week_of_year,cosine_week_of_year,day_of_week,is_public_holiday,num_rentals_t-2,num_rentals_t-3
0,5 Corners Library,2020-01-01,0,0,40.734961,-74.059503,0.0,1.0,2,1,0.0,0.0
1,5 Corners Library,2020-01-01,1,0,40.734961,-74.059503,0.0,1.0,2,1,0.0,0.0
2,5 Corners Library,2020-01-01,2,0,40.734961,-74.059503,0.0,1.0,2,1,0.0,0.0
3,5 Corners Library,2020-01-01,3,0,40.734961,-74.059503,0.0,1.0,2,1,0.0,0.0
4,5 Corners Library,2020-01-01,4,0,40.734961,-74.059503,0.0,1.0,2,1,0.0,0.0


#### 2c. Extract weather features

In [11]:
# get the location and dates to query
weather_search_df = full_rental_counts_df[[STATION, LATITUDE, LONGITUDE, DATE]]
weather_search_df = weather_search_df.drop_duplicates()

start_date = weather_search_df[DATE].min() - dt.timedelta(days = 1)   # 1 extra day so the first hour has a T-1 weather value
end_date = weather_search_df[DATE].max()
stations = set(weather_search_df[STATION])

In [13]:
# call the weather API for each location
extractor = WeatherFeatureExtractor(start_date, end_date)
weather_df = pd.DataFrame()

for i, station in enumerate(stations, start = 1):
    location_df = weather_search_df[weather_search_df[STATION] == station]

    # get historical weather data for the station
    station_weather_history_df = extractor.get_hourly_weather(station, location_df[LATITUDE].iloc[0], location_df[LONGITUDE].iloc[0])
    weather_df = pd.concat([weather_df, station_weather_history_df])

    # pause for 1 min after every 20 calls to stay under the API rate limit
    if i % 20 == 0:
        print(f"{i} stations done, sleeping for 60s...")
        time.sleep(60)

# save results
weather_df.to_csv(data_dir + 'intermediate_processing/historical_weather_features.csv', index=False)

20 stations done, sleeping for 60s...
40 stations done, sleeping for 60s...


In [14]:
weather_df.head()

,date,temperature_2m,rain,snow_depth,snowfall,cloud_cover_low,cloud_cover_mid,cloud_cover_high,relative_humidity_2m,precipitation,wind_speed_10m,wind_gusts_10m,cloud_cover,hour_of_day,start station name,start station latitude,start station longitude
0,2019-12-31,3.10,0.0,0.0,0.0,100.0,1.0,0.0,89.237740,0.0,21.749483,39.239998,100.0,0,Lincoln Park,40.724605,-74.078406
1,2019-12-31,3.00,0.0,0.0,0.0,99.0,2.0,0.0,89.871178,0.0,19.130875,36.000000,99.0,1,Lincoln Park,40.724605,-74.078406
2,2019-12-31,2.80,0.0,0.0,0.0,98.0,2.0,0.0,90.827187,0.0,16.700275,31.680000,98.0,2,Lincoln Park,40.724605,-74.078406
3,2019-12-31,2.75,0.0,0.0,0.0,100.0,3.0,0.0,91.149696,0.0,14.777550,27.359999,100.0,3,Lincoln Park,40.724605,-74.078406
4,2019-12-31,2.20,0.0,0.0,0.0,86.0,24.0,0.0,93.432999,0.0,11.183201,24.119999,88.0,4,Lincoln Park,40.724605,-74.078406


#### 2d. Merge the citibike and weather features

In [15]:
# read previously stored intermediate files
full_rental_counts_df = pd.read_csv(data_dir + 'intermediate_processing/full_rental_counts.csv')
weather_df = pd.read_csv(data_dir + 'intermediate_processing/historical_weather_features.csv')

In [16]:
merge_columns = [STATION, DATE, HOUR_OF_DAY]

# one weather row per station-date-hour
# drops the repeated hour on 1 Nov daylight savings end
weather_to_merge_df = (
    weather_df
    .drop(columns = [LATITUDE, LONGITUDE])
    .drop_duplicates(subset = merge_columns, keep = "first")
)

In [17]:
# lag the weather by 1 hour: the weather recorded at hour T-1 is used to predict hour T
# done on a timestamp so that 11pm rolls over to 12am of the next date
lagged_timestamp = (
    pd.to_datetime(weather_to_merge_df[DATE])
    + pd.to_timedelta(weather_to_merge_df[HOUR_OF_DAY] + 1, unit = 'h')
)
weather_to_merge_df[DATE] = lagged_timestamp.dt.strftime('%Y-%m-%d')
weather_to_merge_df[HOUR_OF_DAY] = lagged_timestamp.dt.hour

# merge features
full_features_df = full_rental_counts_df.merge(
    weather_to_merge_df,
    on = merge_columns,
    how = "left",
    validate = "many_to_one",
)

# 2am doesn't exist on the days the clocks go forward, so 3am has no T-1 weather;
# fill it with the previous available hour (1am) for the same station
weather_cols = [col for col in weather_to_merge_df.columns if col not in merge_columns]
full_features_df = full_features_df.sort_values([STATION, DATE, HOUR_OF_DAY])
full_features_df[weather_cols] = full_features_df.groupby(STATION)[weather_cols].ffill()

# remove the 2am rows on the days the clocks go forward (that hour did not exist)
nonexistent_hour = full_features_df[DATE].isin(['2020-03-08', '2021-03-14']) & (full_features_df[HOUR_OF_DAY] == 2)
full_features_df = full_features_df[~nonexistent_hour]

In [18]:
# drop columns that will not be used
full_features_df = full_features_df.drop(columns=[LATITUDE, LONGITUDE])

# save results
full_features_df.to_csv(data_dir + 'intermediate_processing/full_features.csv', index=False)

In [19]:
full_features_df.head()

,start station name,date,hour_of_day,num_rentals,sine_week_of_year,cosine_week_of_year,day_of_week,is_public_holiday,num_rentals_t-2,num_rentals_t-3,...,snow_depth,snowfall,cloud_cover_low,cloud_cover_mid,cloud_cover_high,relative_humidity_2m,precipitation,wind_speed_10m,wind_gusts_10m,cloud_cover
0,5 Corners Library,2020-01-01,0,0,0.0,1.0,2,1,0.0,0.0,...,0.0,0.0,8.0,53.0,0.0,76.621740,0.0,17.373497,27.359999,54.0
1,5 Corners Library,2020-01-01,1,0,0.0,1.0,2,1,0.0,0.0,...,0.0,0.0,0.0,4.0,0.0,73.943900,0.0,17.673029,29.160000,4.0
2,5 Corners Library,2020-01-01,2,0,0.0,1.0,2,1,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,76.498850,0.0,16.560000,29.160000,0.0
3,5 Corners Library,2020-01-01,3,0,0.0,1.0,2,1,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,78.990500,0.0,16.563911,27.720000,0.0
4,5 Corners Library,2020-01-01,4,0,0.0,1.0,2,1,0.0,0.0,...,0.0,0.0,0.0,0.0,11.0,76.922134,0.0,18.723460,31.680000,11.0


#### 2e. Train-validation-test split and one-hot encoding

In [2]:
full_features_df = pd.read_csv(data_dir + 'intermediate_processing/full_features.csv')

In [3]:
# split into the train-validation and test set
train_validation_df = full_features_df[full_features_df[DATE] <= "2021-02-28"]
test_df = full_features_df[full_features_df[DATE] >= "2021-03-01"]

In [4]:
# do one hot encoding and save the model
processor = FeaturePreprocessor()

# do one hot encoding
y_column = NUM_RENTALS
categorical_cols = [STATION, HOUR_OF_DAY, DAY_OF_WEEK]
non_categorical_cols = [ 
    SINE_WEEK_OF_YEAR, COSINE_WEEK_OF_YEAR, IS_PUBLIC_HOLIDAY, f'{NUM_RENTALS}_t-2', f'{NUM_RENTALS}_t-3',
    TEMPERATURE, RAIN, SNOW_DEPTH, SNOWFALL, CLOUD_COVER, CLOUD_COVER_LOW, CLOUD_COVER_MID, 
    CLOUD_COVER_HIGH, RELATIVE_HUMIDITY, PRECIPITATION, WIND_SPEED, WIND_GUSTS
]

# fit encoder on train set and apply to all sets
processor.fit_one_hot_encoder(categorical_features=train_validation_df[categorical_cols])
encoded_train_validation_df = processor.get_encoded_features_df(train_validation_df, categorical_cols, non_categorical_cols)
encoded_test_df = processor.get_encoded_features_df(test_df, categorical_cols, non_categorical_cols)

# save the fitted encoder to the models folder
processor.save_one_hot_encoder(model_dir + 'one_hot_encoder.pkl')

In [5]:
# add the target variables back in
encoded_train_validation_df[y_column] = train_validation_df[y_column]
encoded_test_df[y_column] = test_df[y_column]

# add date back so that the folds can be created later
encoded_train_validation_df[DATE] = train_validation_df[DATE]
encoded_test_df[DATE] = test_df[DATE]

# save to parquet files
save_to_parquet_file(encoded_train_validation_df, data_dir + 'processed/train_validation.parquet')
save_to_parquet_file(encoded_test_df, data_dir + 'processed/test.parquet')
print('Total number of rows in train-validation:', len(encoded_train_validation_df))
print('Total number of rows in test:', len(encoded_test_df))


Total number of rows in train-validation: 520149
Total number of rows in test: 74409
